# 2장 1강: AARRR 프레임워크 개요와 단계별 분석 목적 — 실습문제

## 실습 목표

- Ravenstack의 사용자 여정을 AARRR 5단계로 구분할 수 있다.
- 각 단계의 분석 목적과 활용 가능한 데이터를 연결할 수 있다.
- 이벤트 로그의 사용자·행동·시점·속성 컬럼을 구분할 수 있다.
- 분석 질문에 맞게 퍼널·코호트·세그멘테이션 분석을 선택할 수 있다.
- 그룹별 Activation 지표를 비교하여 점검 대상을 찾을 수 있다.

## 실습 환경 / 데이터

- Python
- pandas
- `ravenstack_accounts.csv`
- `ravenstack_subscriptions.csv`
- `ravenstack_feature_usage.csv`

Ravenstack은 기업 고객에게 구독형 소프트웨어를 제공하는 B2B SaaS 서비스입니다.

Ravenstack의 AARRR 단계는 다음과 같이 연결할 수 있습니다.

| 단계 | Ravenstack에서의 의미 | 활용 데이터 예시 |
|---|---|---|
| Acquisition | 고객사가 유입되어 가입함 | 가입일, 유입 경로 |
| Activation | 구독 후 제품 기능을 처음 사용함 | 기능 사용 기록 |
| Retention | 제품 사용과 구독을 계속 유지함 | 반복 사용 기록, 이탈 여부 |
| Referral | 기존 고객이 다른 고객을 추천함 | 추천 전송·추천 가입 이벤트 |
| Revenue | 유료 구독에서 반복 매출이 발생함 | MRR, 요금제 |

> 제공된 데이터에는 고객 추천 행동을 직접 기록한 이벤트가 없습니다. `referral_source`는 고객의 유입 경로이며, 기존 고객이 추천 행동을 했다는 사실을 직접 보여주는 Referral 이벤트와는 구분해야 합니다.

## 실습 준비

1. 필요한 라이브러리를 불러오세요.
2. 세 CSV 파일을 각각 `accounts`, `subscriptions`, `feature_usage`에 불러오세요.
3. 각 데이터의 행과 열 개수, 상위 5개 행을 확인하세요.
4. 분석 대상 컬럼의 자료형과 결측치 개수를 확인하세요.
5. `signup_date`, `start_date`, `end_date`, `usage_date`를 날짜형으로 변환하세요.
6. `feature_usage`에서 이벤트 식별자인 `usage_id`의 중복 개수를 확인하세요.
7. 기능별 이벤트 기록 수와 `usage_count`의 기초 통계량을 확인하세요.

In [2]:
import pandas as pd

accounts = pd.read_csv('./ravenstack_accounts.csv')
subscriptions = pd.read_csv('./ravenstack_subscriptions.csv')
feature_usage = pd.read_csv('./ravenstack_feature_usage.csv')

print(accounts.shape, subscriptions.shape, feature_usage.shape)
display(accounts.head())
display(subscriptions.head())
display(feature_usage.head())

print(accounts.dtypes)
print(accounts.isna().sum())
print(subscriptions.dtypes)
print(subscriptions.isna().sum())
print(feature_usage.dtypes)
print(feature_usage.isna().sum())

accounts['signup_date'] = pd.to_datetime(accounts['signup_date'])
subscriptions['start_date'] = pd.to_datetime(subscriptions['start_date'])
subscriptions['end_date'] = pd.to_datetime(subscriptions['end_date'])
feature_usage['usage_date'] = pd.to_datetime(feature_usage['usage_date'])

print('usage_id 중복 개수:', feature_usage['usage_id'].duplicated().sum())

print(feature_usage['feature_name'].value_counts())
print(feature_usage['usage_count'].describe())


(500, 10) (5000, 14) (25000, 8)


,account_id,account_name,industry,country,signup_date,referral_source,plan_tier,seats,is_trial,churn_flag
0,A-2e4581,Company_0,EdTech,US,2024-10-16,partner,Basic,9,False,False
1,A-43a9e3,Company_1,FinTech,IN,2023-08-17,other,Basic,18,False,True
2,A-0a282f,Company_2,DevTools,US,2024-08-27,organic,Basic,1,False,False
3,A-1f0ac7,Company_3,HealthTech,UK,2023-08-27,other,Basic,24,True,False
4,A-ce550d,Company_4,HealthTech,US,2024-10-27,event,Enterprise,35,False,True


,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag
0,S-8cec59,A-3c1a3f,2023-12-23,2024-04-12,Enterprise,14,2786,33432,False,False,False,True,monthly,True
1,S-0f6f44,A-9b9fe9,2024-06-11,NaN,Pro,17,833,9996,False,False,False,False,monthly,True
2,S-51c0d1,A-659280,2024-11-25,NaN,Enterprise,62,0,0,True,True,False,False,annual,False
3,S-f81687,A-e7a1e2,2024-11-23,2024-12-13,Enterprise,5,995,11940,False,False,False,True,monthly,True
4,S-cff5a2,A-ba6516,2024-01-10,NaN,Enterprise,27,5373,64476,False,False,False,False,monthly,True


,usage_id,subscription_id,usage_date,feature_name,usage_count,usage_duration_secs,error_count,is_beta_feature
0,U-1c6c24,S-0fcf7d,2023-07-27,feature_20,9,5004,0,False
1,U-f07cb8,S-c25263,2023-08-07,feature_5,9,369,0,False
2,U-096807,S-f29e7f,2023-12-07,feature_3,9,1458,0,False
3,U-6b1580,S-be655e,2024-07-28,feature_40,5,2085,0,False
4,U-720a29,S-f9b1d0,2024-12-02,feature_12,12,900,0,False


account_id           str
account_name         str
industry             str
country              str
signup_date          str
referral_source      str
plan_tier            str
seats              int64
is_trial            bool
churn_flag          bool
dtype: object
account_id         0
account_name       0
industry           0
country            0
signup_date        0
referral_source    0
plan_tier          0
seats              0
is_trial           0
churn_flag         0
dtype: int64
subscription_id        str
account_id             str
start_date             str
end_date               str
plan_tier              str
seats                int64
mrr_amount           int64
arr_amount           int64
is_trial              bool
upgrade_flag          bool
downgrade_flag        bool
churn_flag            bool
billing_frequency      str
auto_renew_flag       bool
dtype: object
subscription_id         0
account_id              0
start_date              0
end_date             4514
plan_tier        

---

## 필수 1. Ravenstack의 AARRR 지표 지도 만들기

### 문제 1-1. 데이터에서 확인 가능한 단계별 대표 지표 계산하기

#### 문제 설명

Ravenstack의 세 테이블을 이용하여 AARRR 단계별 대표 지표를 계산하고, 제공된 데이터만으로 직접 측정하기 어려운 단계도 확인하세요.

이번 문제에서는 다음 기준을 사용합니다.

- Acquisition: 가입한 고유 고객사 수
- Activation: 전체 구독 중 기능 사용 기록이 한 번 이상 있는 구독의 비율
- Retention: 전체 구독 중 현재 `churn_flag`가 `False`인 구독의 비율
- Referral: 제공된 데이터로 직접 계산할 수 없음
- Revenue: 종료일이 없고 체험 구독이 아닌 현재 활성 유료 구독의 MRR 합계

> 여기에서 Retention은 학습을 위한 간단한 대리 지표입니다. 특정 기간 후 다시 사용한 비율을 계산한 정식 리텐션과는 다릅니다.

#### 요구사항

1. 가입한 고유 고객사 수를 `acquired_accounts`로 계산하세요.
2. 기능 사용 기록이 있는 고유 구독 수를 전체 구독 수로 나누어 `activation_rate`를 계산하세요.
3. `churn_flag`가 `False`인 구독의 비율을 `non_churned_rate`로 계산하세요.
4. 종료일이 없고 체험 구독이 아닌 구독의 MRR 합계를 `current_paid_mrr`로 계산하세요.
5. 네 지표를 알아보기 쉬운 형식으로 출력하세요.
6. 각 지표를 AARRR 단계와 연결하고 분석 목적을 설명하세요.
7. Referral 단계를 직접 측정하려면 어떤 이벤트가 추가로 필요한지 제안하세요.

#### 해석 질문

**Q1.** 기능 사용 기록이 있는 구독 비율은 어느 AARRR 단계와 연결되나요?  
**Q2.** `non_churned_rate`를 정식 리텐션율과 동일하게 보면 안 되는 이유는 무엇인가요?  
**Q3.** `referral_source`만으로 Referral 행동을 직접 측정할 수 있나요?  
**Q4.** MRR은 어느 AARRR 단계와 연결되나요?

#### 제출 결과

- AARRR 단계별 대표 지표 계산 코드와 결과
- 각 지표의 단계 및 분석 목적
- Referral 측정에 필요한 이벤트 제안
- Q1~Q4 답변

In [3]:
# Acquisition: 가입한 고유 고객사 수
acquired_accounts = accounts['account_id'].nunique()

# Activation: 기능 사용 기록이 있는 구독 / 전체 구독
used_sub_ids = set(feature_usage['subscription_id'].unique())
activation_rate = subscriptions['subscription_id'].isin(used_sub_ids).mean()

# Retention (대리 지표): churn_flag가 False인 구독 비율
non_churned_rate = (subscriptions['churn_flag'] == False).mean()

# Revenue: 종료일 없고 trial 아닌 구독의 MRR 합
paid_mask = subscriptions['end_date'].isna() & (subscriptions['is_trial'] == False)
current_paid_mrr = subscriptions.loc[paid_mask, 'mrr_amount'].sum()

print('acquired_accounts:', acquired_accounts)
print('activation_rate:', f'{activation_rate*100:.2f}%')
print('non_churned_rate:', f'{non_churned_rate*100:.2f}%')
print('current_paid_mrr:', f'{current_paid_mrr:,}')


acquired_accounts: 500
activation_rate: 99.34%
non_churned_rate: 90.28%
current_paid_mrr: 10,159,608


네 지표를 AARRR 단계에 연결해보면 이렇게 될 것 같다.

- acquired_accounts → Acquisition: 그냥 가입한 고객사 수라서 유입 단계랑 바로 연결된다.
- activation_rate → Activation: 구독 후에 기능을 한 번이라도 써봤는지를 보여주는 지표라서 Activation이다. 
- non_churned_rate → Retention: 지금 이탈 안 한 구독 비율이라서 Retention 쪽에 가깝다. 근데 이건 "일정 기간 뒤에 다시 썼는지"를 본 게 아니라 지금 시점의 이탈 여부만 본 거라 정식 리텐션이랑은 다르다.
- current_paid_mrr → Revenue 반복 매출을 그대로 보여주는 값이라서 Revenue다.
- Referral은 지금 데이터로는 직접 계산이 안 된다. referral_source는 그냥 "어디서 왔는지"를 나타내는 유입 경로일 뿐, 기존 고객이 실제로 추천 행동을 했다는 걸 보여주는 이벤트가 아니기 때문이다. Referral을 직접 재려면 추천 링크 클릭, 추천 코드 입력 가입 같은 **추천 관련 이벤트 로그**가 따로 필요할 것 같다.


### 필수 1 답변 작성란

- **Q1.** 기능 사용 기록이 있는 구독 비율(`activation_rate`)은 **Activation** 단계와 연결된다. 구독을 시작한 뒤에 실제로 제품을 써봤는지를 보여주는 지표라서 그렇다.
- **Q2.** `non_churned_rate`는 지금 이 시점에 이탈했는지 여부만 본 값이다. 정식 리텐션은 보통 특정 시점 이후에 일정 기간이 지나서 다시 돌아와 사용했는지를 보는데, 이 지표는 그런 시간 흐름을 반영하지 않은 단순 스냅샷이라서 정식 리텐션율이랑 똑같이 취급하면 안 된다고 생각한다.
- **Q3.** 안 된다고 생각한다. `referral_source`는 고객사가 어떤 경로로 들어왔는지를 나타낼 뿐이지, 그 고객사가 실제로 다른 고객을 추천했다는 행동을 기록한 게 아니다. Referral 행동을 직접 보려면 추천 링크 클릭이나 추천을 통한 가입 완료 같은 이벤트가 따로 있어야 할 것 같다.
- **Q4.** MRR(`current_paid_mrr`)은 **Revenue** 단계와 연결된다. 반복해서 들어오는 매출을 그대로 보여주는 값이기 때문이다.


---

## 필수 2. 요금제별 Activation 지표 비교하기

### 문제 2-1. 2024년 12월 제품 활성률을 요금제별로 비교하기

#### 문제 설명

프로덕트팀은 다음 질문에 답하려고 합니다.

> 2024년 12월에 어떤 요금제의 제품 활성률이 상대적으로 낮은가?

이 질문은 사용자를 요금제별로 나누어 비교하므로 **세그멘테이션 분석**에 해당하며, AARRR 중 Activation 단계를 살펴봅니다.

이번 실습에서 제품 활성률은 다음과 같이 정의합니다.

> 12월 제품 활성 유료 구독 수 ÷ 12월 이용 가능 유료 구독 수

#### 요구사항

1. 2024년 12월의 시작일과 마지막 날을 설정하세요.
2. 12월에 이용 가능한 유료 구독을 `eligible_dec`에 저장하세요.
3. 12월에 기능을 사용한 구독 ID를 `active_subscription_ids`로 준비하세요.
4. `eligible_dec`에 `is_product_active` 컬럼을 만드세요.
5. `plan_tier`별로 다음 값을 집계하여 `plan_activation`을 만드세요.
   - 이용 가능 유료 구독 수
   - 제품 활성 유료 구독 수
   - 제품 활성률
6. 제품 활성률이 낮은 순서로 정렬하고 백분율로 출력하세요.
7. 가장 낮은 요금제를 찾아 개선을 위해 추가로 확인할 내용을 제안하세요.
8. 이 질문에 세그멘테이션 분석이 적절한 이유를 설명하세요.

#### 해석 질문

**Q1.** 이 분석은 AARRR 중 어느 단계에 해당하나요?  
**Q2.** 어떤 분석 기법을 사용했으며, 그 이유는 무엇인가요?  
**Q3.** 제품 활성률이 가장 낮은 요금제는 무엇인가요?  
**Q4.** 해당 요금제에서 추가로 어떤 데이터를 확인하면 좋을까요?

#### 제출 결과

- `plan_activation` 집계 코드와 결과
- AARRR 단계와 분석 기법
- 우선 점검할 요금제
- 추가 확인 데이터
- Q1~Q4 답변

In [4]:
dec_start = pd.Timestamp('2024-12-01')
dec_end = pd.Timestamp('2024-12-31')

# 12월에 이용 가능한 유료 구독: 12월 이전에 시작했고, 12월 시작일 이후까지 살아있고, trial이 아닌 구독
eligible_dec = subscriptions[
    (subscriptions['start_date'] <= dec_end) &
    (subscriptions['end_date'].isna() | (subscriptions['end_date'] >= dec_start)) &
    (subscriptions['is_trial'] == False)
].copy()

# 12월에 기능을 사용한 구독 ID
dec_usage = feature_usage[(feature_usage['usage_date'] >= dec_start) & (feature_usage['usage_date'] <= dec_end)]
active_subscription_ids = set(dec_usage['subscription_id'].unique())

eligible_dec['is_product_active'] = eligible_dec['subscription_id'].isin(active_subscription_ids)

plan_activation = eligible_dec.groupby('plan_tier').agg(
    eligible_subscriptions=('subscription_id', 'count'),
    active_subscriptions=('is_product_active', 'sum')
)
plan_activation['activation_rate'] = plan_activation['active_subscriptions'] / plan_activation['eligible_subscriptions']
plan_activation = plan_activation.sort_values('activation_rate')

plan_activation_print = plan_activation.copy()
plan_activation_print['activation_rate'] = (plan_activation_print['activation_rate']*100).round(2).astype(str) + '%'
print(plan_activation_print)


            eligible_subscriptions  active_subscriptions activation_rate
plan_tier                                                               
Pro                           1342                   260          19.37%
Basic                         1267                   247          19.49%
Enterprise                    1372                   281          20.48%


**Pro** 요금제의 제품 활성률이 19.37%로 세 요금제 중에 제일 낮다. Basic(19.49%)이랑은 거의 차이가 없는데 Enterprise(20.48%)보다는 확실히 낮은 편이다.

이 질문은 사용자를 요금제라는 미리 정해진 그룹으로 나눠서 비교한 거니까 **세그멘테이션 분석**이 맞는 방법인 것 같다. 전체 활성률만 봤으면 Pro가 상대적으로 낮다는 걸 몰랐을 텐데, 나눠보니까 바로 보인다.

Pro가 왜 낮은지 더 보려면 Pro 사용자들이 어떤 기능을 안 쓰는지, 온보딩 과정에서 어디서 이탈하는지 같은 걸 추가로 확인해보면 좋을 것 같다.


### 필수 2 답변 작성란

- **Q1.** **Activation** 단계에 해당한다. 구독 후에 제품을 실제로 썼는지를 보는 거라서 그렇다.
- **Q2.** **세그멘테이션 분석**을 썼다. 사용자를 요금제라는 그룹으로 나눠서 그룹 간 차이를 비교했기 때문이다.
- **Q3.** 제품 활성률이 가장 낮은 요금제는 **Pro**(19.37%)다.
- **Q4.** Pro 사용자들이 구체적으로 어떤 기능을 안 쓰는지, 온보딩 단계에서 어디서 막히는지, 혹은 Pro 요금제 특유의 문제(가격/기능 구성 등)가 있는지를 추가로 확인해보면 좋을 것 같다.


---

## 과제 1. 평가 문항 기반 독립 과제

### 문제 3-1. 결제 주기별 Activation 지표 비교하기

#### 문제 설명

프로덕트팀은 2024년 12월 제품 활성률이 월간 결제 구독과 연간 결제 구독에서 다르게 나타나는지 확인하려고 합니다.

> 이 과제는 필수 2의 요금제별 세그멘테이션을 `billing_frequency`에 동일하게 적용하는 문제입니다.

#### 요구사항

1. 필수 2에서 만든 `eligible_dec`과 `is_product_active`를 사용하세요.
2. `billing_frequency`별로 다음 값을 집계하여 `billing_activation`을 만드세요.
   - 이용 가능 유료 구독 수
   - 제품 활성 유료 구독 수
   - 제품 활성률
3. 제품 활성률이 낮은 순서로 정렬하고 백분율로 출력하세요.
4. 제품 활성률이 더 낮은 결제 주기를 찾으세요.
5. 해당 그룹을 우선 점검 대상으로 정하고 추가로 확인할 내용을 한 가지 제안하세요.
6. 이 분석의 AARRR 단계와 분석 기법을 설명하세요.

#### 해석 질문

**Q1.** 제품 활성률이 더 낮은 결제 주기는 무엇인가요?  
**Q2.** 이 분석은 어떤 AARRR 단계와 분석 기법에 해당하나요?  
**Q3.** 두 그룹의 제품 활성률 차이만으로 결제 주기가 낮은 활성의 원인이라고 결론 내릴 수 있나요?

**Q4.** 퍼널 분석·코호트 리텐션 분석·세그멘테이션을 표로 정리하세요. 각 분석의 계산 또는 분류 원리, 해결하려는 분석 질문, 관련 AARRR 단계를 설명하세요. 전환율·이탈률의 분모, 코호트의 시작 기준과 경과 기간, 세그먼트 분류 기준을 포함하세요.

#### 제출 결과

- `billing_activation` 집계 코드와 결과
- 우선 점검할 결제 주기
- 추가 확인 내용
- AARRR 단계와 분석 기법
- Q1~Q4 답변

In [5]:
# 필수 2에서 만든 eligible_dec, is_product_active를 그대로 사용
billing_activation = eligible_dec.groupby('billing_frequency').agg(
    eligible_subscriptions=('subscription_id', 'count'),
    active_subscriptions=('is_product_active', 'sum')
)
billing_activation['activation_rate'] = billing_activation['active_subscriptions'] / billing_activation['eligible_subscriptions']
billing_activation = billing_activation.sort_values('activation_rate')

billing_activation_print = billing_activation.copy()
billing_activation_print['activation_rate'] = (billing_activation_print['activation_rate']*100).round(2).astype(str) + '%'
print(billing_activation_print)


                   eligible_subscriptions  active_subscriptions  \
billing_frequency                                                 
annual                               1968                   385   
monthly                              2013                   403   

                  activation_rate  
billing_frequency                  
annual                     19.56%  
monthly                    20.02%  


연간 결제가 19.56%로 월간 결제보다 조금 낮다. 차이가 0.5%p도 안 되는 수준이라 크다고 하기는 어려운데, 그래도 낮은 쪽인 annual을 우선 점검 대상으로 잡았다. 연간 결제 고객은 한 번 결제하고 나면 매달 결제 화면을 볼 일이 없어서 오히려 제품을 덜 들여다보게 되는 건 아닌지 추가로 확인해보면 좋을 것 같다.

이 분석도 요금제 비교랑 똑같이 Activation 단계를 다룬 세그멘테이션 분석이다.


### 과제 1 답변 작성란

- **Q1.** 제품 활성률이 더 낮은 결제 주기는 annual이다. monthly는 20.02%로 조금 더 높다.
- **Q2.** Activation 단계를 세그멘테이션 분석으로 본 것이다. 결제 주기라는 그룹으로 나눠서 비교했기 때문이다.
- **Q3.** 아니라고 생각한다. 두 그룹의 차이가 0.5%p 정도로 크지 않아서 우연한 차이일 수도 있고, 활성률이 낮은 원인이 결제 주기 자체 때문인지 다른 요인 때문인지는 이 결과만으로는 확정할 수 없다.
- **Q4.**

| 분석 | 계산/분류 원리 | 해결하려는 질문 | 관련 AARRR 단계 |
|---|---|---|---|
| 퍼널 분석 | 정해진 순서의 단계를 지나면서 각 단계 고유 사용자(구독) 수를 세고, 전환율 = 다음 단계 수 ÷ 이전 단계 수로 계산. 분모는 이전 단계를 통과한 사용자 수다. | 어느 단계에서 사람들이 제일 많이 빠지는가? | Activation ~ Revenue 전반 |
| 코호트 리텐션 분석 | 같은 시점(예: 가입 월)에 시작한 사용자를 하나의 코호트로 묶고, 그 코호트가 시작 시점부터 일정 기간(1주/1개월 등)이 지난 뒤에도 남아있는 비율을 추적. 시작 기준은 코호트 생성 시점(가입일 등), 경과 기간은 그 시점 이후 지난 시간 단위다. | 특정 시점에 들어온 사용자들이 시간이 지나면서 얼마나 남아있는가? | Retention |
| 세그멘테이션 | 사용자를 요금제, 유입 경로, 결제 주기 같은 기준으로 그룹을 나누고, 그룹별로 같은 지표를 계산해서 비교. 세그먼트 분류 기준은 비교하고 싶은 속성(요금제, 결제 주기 등)이다. | 어떤 그룹에서 특히 문제가 있는가? | 모든 단계에 적용 가능 (이번 실습에서는 Activation) |


---

## 실습 마무리

아래 질문에 답하세요.

1. Ravenstack의 사용자 여정을 어떤 AARRR 단계로 나누었나요?
2. 각 단계는 어떤 분석 질문에 답하기 위해 사용했나요?
3. 제공된 데이터에서 직접 측정하기 어려운 단계는 무엇이었나요?
4. 요금제와 결제 주기별 비교에는 어떤 분석 기법을 사용했나요?
5. 분석 결과를 근거로 어떤 대상을 우선 점검했나요?

### 실습 마무리 답변

1. Acquisition(가입), Activation(기능 사용 시작), Retention(사용/구독 유지), Referral(추천), Revenue(반복 매출) 다섯 단계로 나눠봤다.
2. Acquisition은 "고객이 얼마나 들어오는가", Activation은 "들어온 고객이 제품을 실제로 쓰는가", Retention은 "계속 쓰고 있는가", Referral은 "다른 고객을 데려오는가", Revenue는 "돈을 계속 내고 있는가"라는 질문에 답하려고 사용했다.
3. Referral 단계는 제공된 데이터로 직접 측정하기 어려웠다. `referral_source`는 유입 경로일 뿐 실제 추천 행동을 기록한 이벤트가 아니기 때문이다.
4. 요금제와 결제 주기별 비교에는 **세그멘테이션 분석**을 사용했다.
5. 요금제 비교에서는 **Pro**, 결제 주기 비교에서는 **annual**의 제품 활성률이 상대적으로 낮게 나와서 이 두 그룹을 우선 점검 대상으로 잡았다.
